# Module 07 Supplement: SciPy with RNA-seq

Follow along here, or work through it on your own. Feel free to add your own cells and notes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv('data/RNAcountTable_chr1_geneNames.tsv', sep='\t')
df = df.rename(columns={'Gene': 'gene'})

## The Data

### Counts Span Orders of Magnitude

In [ ]:
samples = ['FB_1', 'FB_2', 'HSC_1', 'HSC_2', 'TREG_1', 'TREG_2']

df[samples].describe()

### Log Transform

We add 1 before the log, because the log of 0 is undefined.

In [ ]:
log_counts = np.log2(df.set_index('gene')[samples] + 1)

Genes with no reads in any sample carry no information, so let's drop them.

In [ ]:
expressed = df[samples].sum(axis=1) > 0
log_counts = log_counts[expressed.to_numpy()]
log_counts.shape

## Correlation

### Do Replicates Agree?

In [ ]:
sns.scatterplot(data=log_counts, x='FB_1', y='FB_2', s=10)

In [ ]:
stats.pearsonr(log_counts['FB_1'], log_counts['FB_2'])

### Raw Counts vs. Log Counts

In [ ]:
raw = df.set_index('gene')[samples][expressed.to_numpy()]
stats.pearsonr(raw['FB_1'], raw['FB_2'])

### Pearson and Spearman

In [ ]:
stats.spearmanr(raw['FB_1'], raw['FB_2'])

### Replicates vs. Different Cell Types

In [ ]:
stats.pearsonr(log_counts['FB_1'], log_counts['HSC_1'])

### A Correlation Matrix

Q: Do the replicates cluster together? Which two cell types are the most similar?

In [ ]:
corr = log_counts.corr()

sns.heatmap(corr, annot=True, fmt='.2f', vmin=0.7, vmax=1)

### Practice (5 min)

Let's work on these questions:

1. Make a scatterplot of `HSC_1` against `HSC_2` and compute their correlation.
2. Which pair has the higher correlation, `FB_1` vs. `HSC_1` or `TREG_1` vs. `HSC_1`?
3. Compute the Spearman correlation for the pair in question 2. Is it similar to Pearson's?

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

## T-tests

### Is a Gene Expressed Differently?

Is the gene `Fcmr` expressed differently in `FB` than in `HSC`?

In [ ]:
gene = log_counts.loc['Fcmr']

fb = gene[['FB_1', 'FB_2']]
hsc = gene[['HSC_1', 'HSC_2']]

fb.mean(), hsc.mean()

### A t-test on One Gene

In [ ]:
stats.ttest_ind(fb, hsc)

### Writing a Helper Function

In [ ]:
def compare(gene, a, b):
    values = log_counts.loc[gene]
    return stats.ttest_ind(values[a], values[b]).pvalue

compare('Sell', ['FB_1', 'FB_2'], ['HSC_1', 'HSC_2'])

In [ ]:
compare('Ptprc', ['FB_1', 'FB_2'], ['HSC_1', 'HSC_2'])

### Testing Every Gene

`axis=1` runs the test across the columns of each row.

In [ ]:
result = stats.ttest_ind(
    log_counts[['FB_1', 'FB_2']],
    log_counts[['HSC_1', 'HSC_2']],
    axis=1,
)
pvalues = pd.Series(result.pvalue, index=log_counts.index)

(pvalues < 0.05).sum()

### Practice (5 min)

Let's work on these questions:

1. Use `compare()` to test `Ptprc` between `TREG` and `HSC`. Is it significant?
2. Make a bar plot or a strip plot of `Ptprc` across all six samples to see the pattern.
3. How many genes have a p-value below 0.01? How does that compare with the number you'd expect by chance?

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

## Exercise

Let's explore the RNA-seq data with what we learned.

- Pick a pair of cell types and compute the correlation between their mean log counts
- Test the genes `Fcmr`, `Sell`, and `Ptprc` between `FB` and `TREG`
- In a Markdown cell, write a short summary of your findings, including a caution about the number of replicates